# Electric Production — Time Series Modeling & Forecasting

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Load Preprocessed Data

In [ ]:
df = pd.read_csv('Data/Electric_Production_Preprocessed.csv',
                 parse_dates=['DATE'], index_col='DATE')
df = df[['Value']].asfreq('MS')
print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min()} → {df.index.max()}')
df.head()

## 2. Train / Test Split (80% / 20%)

In [ ]:
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
test  = df.iloc[split_idx:]

print(f'Train: {train.index.min().date()} → {train.index.max().date()}  ({len(train)} months)')
print(f'Test : {test.index.min().date()} → {test.index.max().date()}  ({len(test)} months)')

plt.plot(train['Value'], label='Train', color='steelblue')
plt.plot(test['Value'],  label='Test',  color='darkorange')
plt.axvline(train.index[-1], color='red', linestyle='--', linewidth=1)
plt.title('Train / Test Split')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Evaluation Metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate(actual, predicted, model_name):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    print(f'{model_name:<30} MAE={mae:.3f}  RMSE={rmse:.3f}  MAPE={mape:.2f}%')
    return {'Model': model_name, 'MAE': round(mae,3), 'RMSE': round(rmse,3), 'MAPE': round(mape,2)}

results = []

## 4. Baseline — Seasonal Naive

In [ ]:
# Predict each test month using the same month from the previous year
naive_preds = pd.Series(
    (train['Value'].values[-12:].tolist() * (len(test) // 12 + 1))[:len(test)],
    index=test.index
)

results.append(evaluate(test['Value'], naive_preds, 'Seasonal Naive'))

plt.plot(test['Value'],  label='Actual',         color='steelblue')
plt.plot(naive_preds,    label='Seasonal Naive',  color='darkorange', linestyle='--')
plt.title('Baseline — Seasonal Naive Forecast')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. SARIMA — Auto Order Selection

In [ ]:
import pmdarima as pm

auto_model = pm.auto_arima(
    train['Value'],
    seasonal=True, m=12,
    d=1, D=1,
    stepwise=True,
    information_criterion='aic',
    trace=True,
    error_action='ignore',
    suppress_warnings=True
)
print('\nBest order:', auto_model.order)
print('Best seasonal order:', auto_model.seasonal_order)

In [ ]:
sarima_preds = auto_model.predict(n_periods=len(test))
sarima_preds = pd.Series(sarima_preds, index=test.index)

results.append(evaluate(test['Value'], sarima_preds, 'SARIMA (Auto)'))

plt.plot(train['Value'],  label='Train',          color='steelblue',  alpha=0.5)
plt.plot(test['Value'],   label='Actual',          color='steelblue')
plt.plot(sarima_preds,    label='SARIMA Forecast', color='darkorange', linewidth=2)
plt.title('SARIMA Forecast vs Actual')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Prophet

In [ ]:
from prophet import Prophet

# Prophet requires columns named 'ds' and 'y'
train_prophet = train.reset_index().rename(columns={'DATE': 'ds', 'Value': 'y'})

prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                        daily_seasonality=False, seasonality_mode='multiplicative')
prophet_model.fit(train_prophet)

future = prophet_model.make_future_dataframe(periods=len(test), freq='MS')
forecast = prophet_model.predict(future)

prophet_preds = forecast.set_index('ds')['yhat'].loc[test.index]

results.append(evaluate(test['Value'], prophet_preds, 'Prophet'))

plt.plot(test['Value'],   label='Actual',           color='steelblue')
plt.plot(prophet_preds,   label='Prophet Forecast',  color='green', linewidth=2)
plt.title('Prophet Forecast vs Actual')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. XGBoost with Lag Features

In [ ]:
from xgboost import XGBRegressor

def create_features(data):
    d = data.copy()
    # Calendar features
    d['month']   = d.index.month
    d['quarter'] = d.index.quarter
    d['year']    = d.index.year
    # Lag features
    for lag in [1, 6, 12, 24]:
        d[f'lag_{lag}'] = d['Value'].shift(lag)
    # Rolling features
    d['rolling_mean_12'] = d['Value'].shift(1).rolling(12).mean()
    d['rolling_std_12']  = d['Value'].shift(1).rolling(12).std()
    return d

df_feat = create_features(df).dropna()
feature_cols = [c for c in df_feat.columns if c != 'Value']

X_train = df_feat.loc[train.index.intersection(df_feat.index), feature_cols]
y_train = df_feat.loc[train.index.intersection(df_feat.index), 'Value']
X_test  = df_feat.loc[test.index.intersection(df_feat.index),  feature_cols]
y_test  = df_feat.loc[test.index.intersection(df_feat.index),  'Value']

xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05,
                          max_depth=4, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = pd.Series(xgb_model.predict(X_test), index=y_test.index)

results.append(evaluate(y_test, xgb_preds, 'XGBoost'))

plt.plot(y_test,     label='Actual',           color='steelblue')
plt.plot(xgb_preds,  label='XGBoost Forecast',  color='purple', linewidth=2)
plt.title('XGBoost Forecast vs Actual')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).set_index('Model')
print('=== Model Comparison ===')
print(results_df.to_string())
print(f'\nBest model by RMSE : {results_df["RMSE"].idxmin()}')
print(f'Best model by MAPE : {results_df["MAPE"].idxmin()}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    results_df[metric].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(metric, fontweight='bold')
    ax.set_xticklabels(results_df.index, rotation=30, ha='right')
    ax.grid(True, alpha=0.3, axis='y')
plt.suptitle('Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Best Model — Future Forecast (12 months)

In [ ]:
# Refit best model (SARIMA) on full data and forecast 12 months ahead
best_order          = auto_model.order
best_seasonal_order = auto_model.seasonal_order

final_model = pm.ARIMA(order=best_order, seasonal_order=best_seasonal_order)
final_model.fit(df['Value'])

future_preds, conf_int = final_model.predict(n_periods=12, return_conf_int=True)
future_index = pd.date_range(start=df.index[-1] + pd.DateOffset(months=1),
                              periods=12, freq='MS')
future_series = pd.Series(future_preds, index=future_index)

plt.plot(df['Value'].iloc[-48:],  label='Historical',       color='steelblue')
plt.plot(future_series,           label='12-Month Forecast', color='darkorange', linewidth=2)
plt.fill_between(future_index, conf_int[:, 0], conf_int[:, 1],
                 color='darkorange', alpha=0.2, label='95% CI')
plt.title('12-Month Future Forecast (SARIMA)', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nForecast values:')
print(future_series.to_string())

## 10. Interactive Year Forecast

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Last date in the dataset
last_date = df.index[-1]
first_forecast_year = last_date.year + 1  # first full forecastable year

def forecast_for_year(year):
    """Forecast all 12 months of the given year using the SARIMA final_model."""
    target_start = pd.Timestamp(f'{year}-01-01')
    # Number of months from the end of training data to the end of the target year
    months_needed = (target_start.year - last_date.year) * 12 + (target_start.month - last_date.month) + 11
    preds, ci = final_model.predict(n_periods=months_needed, return_conf_int=True)
    idx = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=months_needed, freq='MS')
    full_series = pd.Series(preds, index=idx)
    full_ci     = pd.DataFrame(ci, index=idx, columns=['lower', 'upper'])
    year_series = full_series[full_series.index.year == year]
    year_ci     = full_ci[full_ci.index.year == year]
    return year_series, year_ci

# --- Widgets ---
year_slider = widgets.IntSlider(
    value=first_forecast_year,
    min=first_forecast_year,
    max=first_forecast_year + 9,
    step=1,
    description='Year:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

year_input = widgets.BoundedIntText(
    value=first_forecast_year,
    min=first_forecast_year,
    max=first_forecast_year + 9,
    description='Or type:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

# Keep slider and text box in sync
widgets.jslink((year_slider, 'value'), (year_input, 'value'))

out = widgets.Output()

def on_year_change(change):
    with out:
        clear_output(wait=True)
        year = change['new']
        preds, ci = forecast_for_year(year)

        # Confidence metrics
        ci_width      = (ci['upper'] - ci['lower'])
        rel_width     = (ci_width / preds.values) * 100          # CI width as % of forecast
        precision     = 100 - rel_width                           # higher = tighter = more precise
        avg_ci_width  = ci_width.mean()
        avg_precision = precision.mean()
        min_prec_month = preds.index[precision.argmin()].strftime('%B')  # least confident month
        max_prec_month = preds.index[precision.argmax()].strftime('%B')  # most confident month

        # Table
        result_df = pd.DataFrame({
            'Month'      : preds.index.strftime('%B'),
            'Forecast'   : preds.values.round(2),
            'Lower 95%'  : ci['lower'].values.round(2),
            'Upper 95%'  : ci['upper'].values.round(2),
            'CI Width'   : ci_width.values.round(2),
            'Precision %': precision.values.round(1),
        })
        print(f'\n  SARIMA Forecast — {year}  (Index: 2017 = 100)')
        print('  ' + '='*65)
        print(result_df.to_string(index=False))
        print(f'\n  --- Summary ---')
        print(f'  Annual Average Forecast : {preds.mean():.2f}')
        print(f'  Annual Total Forecast   : {preds.sum():.2f}')
        print(f'  Avg CI Width            : ±{avg_ci_width/2:.2f} index points')
        print(f'  Avg Precision           : {avg_precision:.1f}%')
        print(f'  Most confident month    : {max_prec_month}  ({precision.max():.1f}%)')
        print(f'  Least confident month   : {min_prec_month}  ({precision.min():.1f}%)')

        # --- Plots ---
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        # Left: forecast + CI band
        ax1 = axes[0]
        ax1.plot(preds.index, preds.values, marker='o', color='darkorange', linewidth=2, label='Forecast')
        ax1.fill_between(preds.index, ci['lower'], ci['upper'], color='darkorange', alpha=0.2, label='95% CI')
        ax1.set_title(f'Monthly Forecast — {year}', fontweight='bold')
        ax1.set_xticks(preds.index)
        ax1.set_xticklabels(preds.index.strftime('%b'), rotation=45)
        ax1.set_ylabel('Electric Production Index (2017=100)')
        ax1.grid(True, alpha=0.3)
        ax1.legend()

        # Right: precision per month
        ax2 = axes[1]
        colors = ['#2ecc71' if p >= avg_precision else '#e74c3c' for p in precision]
        ax2.bar(preds.index.strftime('%b'), precision.values, color=colors, edgecolor='white')
        ax2.axhline(avg_precision, color='steelblue', linestyle='--', linewidth=1.5, label=f'Avg {avg_precision:.1f}%')
        ax2.set_title(f'Forecast Precision by Month — {year}', fontweight='bold')
        ax2.set_ylabel('Precision % (higher = more confident)')
        ax2.set_ylim(max(0, precision.min() - 5), min(100, precision.max() + 5))
        ax2.grid(True, alpha=0.3, axis='y')
        ax2.legend()

        plt.tight_layout()
        plt.show()

year_slider.observe(on_year_change, names='value')

display(widgets.HBox([year_slider, year_input]))
display(out)

# Trigger initial display
on_year_change({'new': year_slider.value})